# 🧠 แผนที่ลักษณะ (Feature Maps) ใน CNN: ช่องทางการกระตุ้นเชิงพื้นที่ (Spatial Activation Channels)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Feature Maps ใน CNNs**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายมิติเทนเซอร์ $[C, H, W]$ ของแผนที่ลักษณะ และวิธีที่ช่องสัญญาณ (channels) ต่างๆ สกัดลักษณะทางทัศนศิลป์ที่แตกต่างกันออกไป
2. สร้างรูปภาพจำลองที่มีรูปทรงต่างๆ (วงกลมและสี่เหลี่ยม) โดยใช้ NumPy
3. กำหนดตัวกรอง Sobel แนวนอน, Sobel แนวตั้ง และตัวกรองปรับความคมชัด (sharpening filter)
4. ทำคอนโวลูชันรูปภาพด้วยตัวกรองเหล่านี้เพื่อสร้าง **ช่องสัญญาณแผนที่ลักษณะที่แตกต่างกัน 3 ช่อง (three distinct feature map channels)**
5. สร้างเลเยอร์ PyTorch `nn.Conv2d` โหลดตัวกรองที่กำหนดเองลงในเทนเซอร์น้ำหนัก (weight tensors) และสกัดแผนที่ลักษณะแบบหลายช่องสัญญาณโดยอัตโนมัติ
6. พล็อตกราฟแผนที่ลักษณะเพื่อแสดงภาพว่าเลเยอร์ CNN มองเห็นขอบและพื้นผิว (textures) อย่างไร
7. เชื่อมโยงแผนที่ลักษณะเข้ากับเลเยอร์การทำนายแบบหลายระดับ (multi-scale prediction layers) ของ YOLO (P3, P4, P5)

เรามาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้างรูปภาพรูปทรงจำลอง

เราจะวาดรูปสี่เหลี่ยมและเส้นทแยงมุมบนผืนผ้าใบสีดำขนาด $64 \times 64$ พิกเซล เพื่อสร้างลักษณะให้ตัวกรองของเราสกัดออกมาได้ครับ

In [ ]:
img = np.zeros((64, 64))

# Draw a square
img[15:45, 15:45] = 1.0

# Draw a diagonal line
for i in range(10, 55):
    img[i, 54 - i] = 1.0

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap='gray')
plt.title('Original Image')
plt.axis('off')
plt.show()

## 2. การกำหนดตัวกรองเฉพาะ (Custom Kernels / Filters)

เรากำหนดตัวกรอง (kernels) สามแบบดังนี้:
1.  **ตัวตรวจจับขอบแนวนอน (Sobel Y):** ตรวจจับขอบเขตแนวนอน
2.  **ตัวตรวจจับขอบแนวตั้ง (Sobel X):** ตรวจจับขอบเขตแนวตั้ง
3.  **ตัวกรองปรับความคมชัด (Sharpen Filter):** เพิ่มความคมชัดโดยรวมของภาพ

In [ ]:
k_horizontal = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
])

k_vertical = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

k_sharpen = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
])

## 3. การโหลดตัวกรองเข้าสู่เลเยอร์ PyTorch Conv2d

เราจะกำหนดเลเยอร์ PyTorch Conv2d ที่มีข้อกำหนดดังนี้:
- `in_channels = 1`
- `out_channels = 3` (แผนที่ลักษณะ 3 ช่องสัญญาณ)
- `kernel_size = 3`

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=3, kernel_size=3, padding=1, bias=False)

weights = np.stack([k_horizontal, k_vertical, k_sharpen])
weights = weights[:, np.newaxis, :, :]

with torch.no_grad():
    conv.weight.copy_(torch.tensor(weights, dtype=torch.float32))

img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

with torch.no_grad():
    feature_maps = conv(img_tensor).squeeze(0)

print("Output Feature Maps Tensor Shape:", feature_maps.shape)

## 4. การแสดงผลภาพช่องสัญญาณแผนที่ลักษณะ

เรามาพล็อตแผนที่ลักษณะทั้ง 3 ช่องสัญญาณข้างกันกันเลยครับ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

titles = [
    "Channel 0: Horizontal Edge Map",
    "Channel 1: Vertical Edge Map",
    "Channel 2: Sharpened Feature Map"
]

for idx in range(3):
    f_map = feature_maps[idx].numpy()
    axes[idx].imshow(f_map, cmap='gray')
    axes[idx].set_title(titles[idx])
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

สังเกตช่องสัญญาณแต่ละช่องกันครับ!
-   **ช่องสัญญาณ 0 (Channel 0):** เส้นแนวนอนด้านบนและด้านล่างของรูปสี่เหลี่ยมจะสว่างเด่นขึ้นมา ส่วนเส้นแนวตั้งจะหายไป
-   **ช่องสัญญาณ 1 (Channel 1):** ขอบแนวตั้งด้านซ้ายและด้านขวาของรูปสี่เหลี่ยมจะถูกเน้นให้เห็นชัดเจน ส่วนเส้นแนวนอนจะหายไป
-   **ช่องสัญญาณ 2 (Channel 2):** ปรับความคมชัดให้สี่เหลี่ยมทั้งรูป ทำให้ขอบเขตต่างๆ เด่นชัดยิ่งขึ้น

## 💡 การเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **พีระมิดแผนที่ลักษณะแบบหลายขนาด (Multi-Scale Feature Pyramid):** YOLO จะดึงข้อมูลแผนที่ลักษณะที่มีหลายระดับการสุ่มจากความลึกที่แตกต่างกันในโครงข่ายประสาทเทียม:
    -   **P3 (Stride 8):** ความละเอียดเชิงพื้นที่สูง (เช่น $80 \times 80$) ซึ่งจะรักษาขอบของวัตถุและแผนที่พิกัดตำแหน่งที่ละเอียดเอาไว้ ใช้ในการตรวจจับวัตถุขนาดเล็ก (เช่น `small-valves` หรือวาล์วขนาดเล็ก)
    -   **P4 (Stride 16):** ความละเอียดระดับปานกลาง (เช่น $40 \times 40$)
    -   **P5 (Stride 32):** ความละเอียดเชิงพื้นที่ต่ำ (เช่น $20 \times 20$) แต่มีระดับช่องสัญญาณความหมายสูง (เช่น 512 ช่องสัญญาณที่แสดงชิ้นส่วนที่ซับซ้อนของวัตถุ) ใช้ในการจำแนกวัตถุขนาดใหญ่
*   การรักษาลำดับขั้นของแผนที่ลักษณะทำให้ YOLO สามารถตรวจจับวัตถุที่มีขนาดแตกต่างกันได้อย่างแม่นยำด้วยการส่งต่อข้อมูลแบบไปข้างหน้าเพียงครั้งเดียว (single forward pass)